Requirement 1: Database Setup
Creating SQLite tables for tests, slots, and bookings.



In [21]:

import sqlite3

# Close the previous connection if it exists
if "conn" in globals():
    conn.close()

conn = sqlite3.connect(
    "lab_booking.db",
    check_same_thread=False
)

conn.execute("PRAGMA foreign_keys = ON")
cursor = conn.cursor()

print("SQLite connection updated successfully!")


SQLite connection updated successfully!


Creating three SQLite tables to store lab tests, appointment slots, and booking details.

In [2]:

cursor.executescript("""
CREATE TABLE IF NOT EXISTS tests (
    test_id INTEGER PRIMARY KEY,
    test_name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL CHECK(price >= 0)
);

CREATE TABLE IF NOT EXISTS slots (
    slot_id INTEGER PRIMARY KEY,
    test_id INTEGER NOT NULL,
    slot_date TEXT NOT NULL,
    slot_time TEXT NOT NULL,
    is_available INTEGER DEFAULT 1
        CHECK(is_available IN (0, 1)),
    FOREIGN KEY (test_id) REFERENCES tests(test_id),
    UNIQUE(test_id, slot_date, slot_time)
);

CREATE TABLE IF NOT EXISTS bookings (
    booking_id INTEGER PRIMARY KEY,
    user_id TEXT NOT NULL,
    test_id INTEGER NOT NULL,
    slot_id INTEGER NOT NULL,
    status TEXT NOT NULL DEFAULT 'confirmed'
        CHECK(status IN ('confirmed', 'cancelled')),
    FOREIGN KEY (test_id) REFERENCES tests(test_id),
    FOREIGN KEY (slot_id) REFERENCES slots(slot_id)
);

CREATE UNIQUE INDEX IF NOT EXISTS unique_active_slot
ON bookings(slot_id)
WHERE status = 'confirmed';
""")

conn.commit()
print("All three tables created successfully!")


All three tables created successfully!


Adding eight lab tests across three categories and creating appointment slots for each test.

In [3]:

from datetime import date, timedelta

# Adding 8 tests across 3 categories
tests_data = [
    (1, "Complete Blood Count", "Blood", 350),
    (2, "Blood Sugar", "Blood", 150),
    (3, "Lipid Profile", "Blood", 600),
    (4, "Thyroid Profile", "Hormonal", 550),
    (5, "Vitamin D", "Hormonal", 900),
    (6, "HbA1c", "Diabetes", 500),
    (7, "Fasting Glucose", "Diabetes", 200),
    (8, "Insulin Test", "Hormonal", 750)
]

cursor.executemany(
    "INSERT OR IGNORE INTO tests VALUES (?, ?, ?, ?)",
    tests_data
)

# Creating 3 slots for each test
start_date = date.today() + timedelta(days=7)
slots_data = []

for test_id in range(1, 9):
    for day in range(3):
        slot_id = test_id * 10 + day + 1
        slot_date = str(start_date + timedelta(days=day))
        slot_time = ["09:00", "10:00", "11:00"][day]

        # Test 8 has no free slots
        available = 0 if test_id == 8 else 1

        slots_data.append(
            (slot_id, test_id, slot_date, slot_time, available)
        )

cursor.executemany(
    "INSERT OR IGNORE INTO slots VALUES (?, ?, ?, ?, ?)",
    slots_data
)

conn.commit()

print("Tests:", cursor.execute(
    "SELECT COUNT(*) FROM tests"
).fetchone()[0])

print("Categories:", cursor.execute(
    "SELECT COUNT(DISTINCT category) FROM tests"
).fetchone()[0])

print("Slots:", cursor.execute(
    "SELECT COUNT(*) FROM slots"
).fetchone()[0])

print("Free slots for Test 8:", cursor.execute(
    "SELECT COUNT(*) FROM slots WHERE test_id = 8 AND is_available = 1"
).fetchone()[0])


Tests: 8
Categories: 3
Slots: 24
Free slots for Test 8: 0


Making cancellation and rebooking possible

###Requirement 2 — Read Tools
Installing LangChain to create tools that the AI agent can use to access the database.

In [4]:
%pip install -q langchain

Creating the list_tests Tool to display lab tests, categories, and prices from SQLite

In [5]:

from langchain.tools import tool, ToolRuntime

@tool
def list_tests() -> str:
    """List all available lab tests with their categories and prices."""
    rows = conn.execute(
        "SELECT test_id, test_name, category, price FROM tests ORDER BY test_id"
    ).fetchall()

    return str([
        {
            "test_id": row[0],
            "test_name": row[1],
            "category": row[2],
            "price": row[3]
        }
        for row in rows
    ])

# Testing the LangChain tool
print(list_tests.invoke({}))


[{'test_id': 1, 'test_name': 'Complete Blood Count', 'category': 'Blood', 'price': 350.0}, {'test_id': 2, 'test_name': 'Blood Sugar', 'category': 'Blood', 'price': 150.0}, {'test_id': 3, 'test_name': 'Lipid Profile', 'category': 'Blood', 'price': 600.0}, {'test_id': 4, 'test_name': 'Thyroid Profile', 'category': 'Hormonal', 'price': 550.0}, {'test_id': 5, 'test_name': 'Vitamin D', 'category': 'Hormonal', 'price': 900.0}, {'test_id': 6, 'test_name': 'HbA1c', 'category': 'Diabetes', 'price': 500.0}, {'test_id': 7, 'test_name': 'Fasting Glucose', 'category': 'Diabetes', 'price': 200.0}, {'test_id': 8, 'test_name': 'Insulin Test', 'category': 'Hormonal', 'price': 750.0}]


Creating a LangChain tool to check available appointment slots for a selected lab test.

In [6]:

@tool
def check_slots(test_id: int) -> str:
    """Check available appointment slots for a given test ID."""
    rows = conn.execute(
        """
        SELECT slot_id, slot_date, slot_time
        FROM slots
        WHERE test_id = ? AND is_available = 1
        ORDER BY slot_date, slot_time
        """,
        (test_id,)
    ).fetchall()

    return str([
        {
            "slot_id": row[0],
            "date": row[1],
            "time": row[2]
        }
        for row in rows
    ])


# Testing available and unavailable slots
print("Test 1:", check_slots.invoke({"test_id": 1}))
print("Test 8:", check_slots.invoke({"test_id": 8}))


Test 1: [{'slot_id': 11, 'date': '2026-11-27', 'time': '09:00'}, {'slot_id': 12, 'date': '2026-11-28', 'time': '10:00'}, {'slot_id': 13, 'date': '2026-11-29', 'time': '11:00'}]
Test 8: []


 LangChain tool to retrieve the current user's bookings securely from SQLite.

In [7]:

from dataclasses import dataclass

@dataclass
class UserContext:
    user_id: str

@tool
def my_bookings(runtime: ToolRuntime[UserContext]) -> str:
    """Show bookings belonging to the current user."""
    user_id = runtime.context.user_id

    rows = conn.execute(
        """
        SELECT b.booking_id, t.test_name,
               s.slot_date, s.slot_time, b.status
        FROM bookings b
        JOIN tests t ON b.test_id = t.test_id
        JOIN slots s ON b.slot_id = s.slot_id
        WHERE b.user_id = ?
        ORDER BY s.slot_date, s.slot_time
        """,
        (user_id,)
    ).fetchall()

    return str(rows)


# Check what arguments the AI model can see
print("Tool schema:", my_bookings.tool_call_schema.model_json_schema())


Tool schema: {'description': 'Show bookings belonging to the current user.', 'properties': {}, 'title': 'my_bookings', 'type': 'object'}


Creating a booking tool that checks test availability and prevents duplicate bookings.

In [8]:

@tool
def book_test(test_id: int, slot_id: int,
              runtime: ToolRuntime[UserContext]) -> str:
    """Book a lab test in an available slot for the current user."""
    user_id = runtime.context.user_id

    test = conn.execute(
        "SELECT test_name, price FROM tests WHERE test_id = ?",
        (test_id,)
    ).fetchone()

    if not test:
        return "Test does not exist."

    duplicate = conn.execute(
        """SELECT booking_id FROM bookings
           WHERE user_id = ? AND test_id = ?
           AND status = 'confirmed'""",
        (user_id, test_id)
    ).fetchone()

    if duplicate:
        return "You already have an active booking for this test."

    with conn:
        updated = conn.execute(
            """UPDATE slots SET is_available = 0
               WHERE slot_id = ? AND test_id = ?
               AND is_available = 1""",
            (slot_id, test_id)
        )

        if updated.rowcount == 0:
            return "Slot is unavailable or does not belong to this test."

        booking = conn.execute(
            """INSERT INTO bookings (user_id, test_id, slot_id)
               VALUES (?, ?, ?)""",
            (user_id, test_id, slot_id)
        )

        receipt = conn.execute(
            """SELECT b.booking_id, t.test_name, t.price,
                      s.slot_date, s.slot_time
               FROM bookings b
               JOIN tests t ON b.test_id = t.test_id
               JOIN slots s ON b.slot_id = s.slot_id
               WHERE b.booking_id = ?""",
            (booking.lastrowid,)
        ).fetchone()

    return f"Booking confirmed. Receipt: {receipt}"


print(book_test.tool_call_schema.model_json_schema())


{'description': 'Book a lab test in an available slot for the current user.', 'properties': {'test_id': {'title': 'Test Id', 'type': 'integer'}, 'slot_id': {'title': 'Slot Id', 'type': 'integer'}}, 'required': ['test_id', 'slot_id'], 'title': 'book_test', 'type': 'object'}


Creating a cancellation tool that verifies booking ownership and releases the appointment slot.

In [9]:

@tool
def cancel_booking(
    booking_id: int,
    runtime: ToolRuntime[UserContext]
) -> str:
    """Cancel an active booking belonging to the current user."""
    user_id = runtime.context.user_id

    with conn:
        booking = conn.execute(
            """SELECT slot_id FROM bookings
               WHERE booking_id = ? AND user_id = ?
               AND status = 'confirmed'""",
            (booking_id, user_id)
        ).fetchone()

        if booking is None:
            return "Booking not found or cancellation not allowed."

        conn.execute(
            """UPDATE bookings SET status = 'cancelled'
               WHERE booking_id = ? AND user_id = ?""",
            (booking_id, user_id)
        )

        conn.execute(
            """UPDATE slots SET is_available = 1
               WHERE slot_id = ?""",
            (booking[0],)
        )

    return f"Booking {booking_id} cancelled successfully."


# Check the tool schema without cancelling anything
print(cancel_booking.tool_call_schema.model_json_schema())


{'description': 'Cancel an active booking belonging to the current user.', 'properties': {'booking_id': {'title': 'Booking Id', 'type': 'integer'}}, 'required': ['booking_id'], 'title': 'cancel_booking', 'type': 'object'}


Requirement 4 — User Identity and Tool Schema

Checking that the AI model cannot access or modify the current user's identity through tool arguments.

In [10]:

print("MODEL-VISIBLE TOOL SCHEMAS\n")

for selected_tool in [my_bookings, book_test, cancel_booking]:
    schema = selected_tool.tool_call_schema.model_json_schema()

    print("Tool:", selected_tool.name)
    print("Arguments:", list(schema["properties"].keys()))
    print("User ID exposed:", "user_id" in schema["properties"])
    print("-" * 40)


MODEL-VISIBLE TOOL SCHEMAS

Tool: my_bookings
Arguments: []
User ID exposed: False
----------------------------------------
Tool: book_test
Arguments: ['test_id', 'slot_id']
User ID exposed: False
----------------------------------------
Tool: cancel_booking
Arguments: ['booking_id']
User ID exposed: False
----------------------------------------


Requirement 5 — Web Search Tool

Installing a web search library to retrieve general information about laboratory tests.

In [11]:

%pip install -q ddgs


Creating a LangChain tool to search for general medical test information without accessing booking details.

In [12]:

from ddgs import DDGS

@tool
def web_search(query: str) -> str:
    """Search general information about lab tests, fasting, and what tests measure. Never search for prices or appointment slots."""
    blocked = ["price", "cost", "fee", "slot", "booking", "appointment"]

    if any(word in query.lower() for word in blocked):
        return "Use the SQLite database tools for prices and appointments."

    try:
        results = DDGS().text(query, max_results=3)

        return str([
            {
                "title": r.get("title"),
                "summary": r.get("body"),
                "url": r.get("href")
            }
            for r in results
        ])
    except Exception as e:
        return f"Web search unavailable: {type(e).__name__}"


print("Web search tool created:", web_search.name)


Web search tool created: web_search


Requirement 6 — Creating the AI Agent with Groq

In [13]:


%pip install -q --upgrade langchain langchain-core langchain-groq langgraph


In [15]:

import os
from google.colab import userdata
from langchain_groq import ChatGroq

api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing")

os.environ["GROQ_API_KEY"] = api_key

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

print("Groq model initialized successfully!")


Groq model initialized successfully!


In [16]:

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

print("LangChain and Groq imports successful!")


LangChain and Groq imports successful!


Requirement 6 — Creating the Agent's System Prompt

Setting rules for the AI agent to use database information, protect patient data, and avoid medical diagnosis.

In [17]:

system_prompt = """
You are a helpful lab test booking assistant for a fictional diagnostic lab.

Follow these rules:
1. Use database tools for lab tests, prices, slots and bookings.
2. Never guess test prices or appointment availability.
3. Ask for approval before booking or cancelling a test.
4. Only access bookings belonging to the current user.
5. Use web search only for general lab test information.
6. Never diagnose diseases or recommend tests based on symptoms.
7. For medical symptoms, suggest consulting a qualified healthcare professional.
8. Keep responses clear, short and helpful.
"""

print("Agent system prompt created successfully!")


Agent system prompt created successfully!


Creating the AI agent with Groq, database tools, conversation memory, and approval for booking actions.

In [18]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# Save conversation history across turns
memory = InMemorySaver()

# Approval required only for database changes
approval = HumanInTheLoopMiddleware(
    interrupt_on={
        "book_test": {
            "allowed_decisions": ["approve", "reject"]
        },
        "cancel_booking": {
            "allowed_decisions": ["approve", "reject"]
        }
    }
)

# Create the agent
agent = create_agent(
    model=llm,
    tools=[
        list_tests,
        check_slots,
        my_bookings,
        book_test,
        cancel_booking,
        web_search
    ],
    system_prompt=system_prompt,
    context_schema=UserContext,
    middleware=[approval],
    checkpointer=memory
)

print("Lab booking agent created successfully!")


Lab booking agent created successfully!


Testing whether the AI agent can retrieve lab test details directly from the database.

In [22]:

#Testing the agent with a read-only question
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "List all available lab tests with their prices."
            }
        ]
    },
    config={"configurable": {"thread_id": "user_101_chat"}},
    context=UserContext(user_id="user_101")
)

# Display tool calls and final response
for message in result["messages"]:
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print("Tool called:", call["name"])

print("\nAGENT RESPONSE:")
print(result["messages"][-1].content)


Tool called: list_tests
Tool called: list_tests
Tool called: list_tests

AGENT RESPONSE:
Here are all the lab tests we offer along with their prices:

| Test ID | Test Name          | Category   | Price (₹) |
|---------|--------------------|------------|-----------|
| 1       | Complete Blood Count | Blood      | 350.0     |
| 2       | Blood Sugar        | Blood      | 150.0     |
| 3       | Lipid Profile      | Blood      | 600.0     |
| 4       | Thyroid Profile    | Hormonal   | 550.0     |
| 5       | Vitamin D          | Hormonal   | 900.0     |
| 6       | HbA1c              | Diabetes   | 500.0     |
| 7       | Fasting Glucose    | Diabetes   | 200.0     |
| 8       | Insulin Test       | Hormonal   | 750.0     |

Let me know if you’d like to book any of these tests or need more details!


Requirement 7 — Approval and Rejection

In [23]:

# Check database state before booking
before = conn.execute(
    "SELECT COUNT(*) FROM bookings WHERE status = 'confirmed'"
).fetchone()[0]

print("Confirmed bookings before:", before)

# Request a booking without approving it
booking_result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Book Complete Blood Count for slot 11. Test ID is 1."
        }]
    },
    config={"configurable": {"thread_id": "approval_test_101"}},
    context=UserContext(user_id="user_101")
)

# Inspect pending approval requests
print("\nPending interrupts:")
print(booking_result.get("__interrupt__", []))

# Confirm that the database has not changed
after = conn.execute(
    "SELECT COUNT(*) FROM bookings WHERE status = 'confirmed'"
).fetchone()[0]

print("\nConfirmed bookings after:", after)
print("Database unchanged:", before == after)


Confirmed bookings before: 0

Pending interrupts:
[]

Confirmed bookings after: 0
Database unchanged: True


Checking why the agent did not trigger the booking approval request.

In [24]:

print("AGENT MESSAGES:\n")

for message in booking_result["messages"]:
    print("Type:", message.type)

    if getattr(message, "tool_calls", None):
        print("Tool calls:", message.tool_calls)

    if message.content:
        print("Content:", str(message.content)[:1000])

    print("-" * 50)

print("\nPending approval:")
print(booking_result.get("__interrupt__", []))


AGENT MESSAGES:

Type: human
Content: Book Complete Blood Count for slot 11. Test ID is 1.
--------------------------------------------------
Type: ai
Tool calls: [{'name': 'check_slots', 'args': {'test_id': 1}, 'id': 'fc_9eb20b36-de69-492d-8317-4ebe6a642831', 'type': 'tool_call'}]
--------------------------------------------------
Type: tool
Content: [{'slot_id': 11, 'date': '2026-11-27', 'time': '09:00'}, {'slot_id': 12, 'date': '2026-11-28', 'time': '10:00'}, {'slot_id': 13, 'date': '2026-11-29', 'time': '11:00'}]
--------------------------------------------------
Type: ai
Content: Sure! The Complete Blood Count (CBC) is available in slot 11 on 2026‑11‑27 at 09:00.  
Would you like me to book this slot for you?
--------------------------------------------------

Pending approval:
[]


In [25]:

# Continue the same conversation
booking_confirm = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Yes, please book slot 11 for me."
        }]
    },
    config={"configurable": {"thread_id": "approval_test_101"}},
    context=UserContext(user_id="user_101")
)

# Display pending approval
print("Pending approval:")
print(booking_confirm.get("__interrupt__", []))

# Verify no booking has been created yet
count = conn.execute(
    "SELECT COUNT(*) FROM bookings WHERE status = 'confirmed'"
).fetchone()[0]

print("\nConfirmed bookings:", count)


Pending approval:
[Interrupt(value={'action_requests': [{'name': 'book_test', 'args': {'slot_id': 11, 'test_id': 1}, 'description': "Tool execution requires approval\n\nTool: book_test\nArgs: {'slot_id': 11, 'test_id': 1}"}], 'review_configs': [{'action_name': 'book_test', 'allowed_decisions': ['approve', 'reject']}]}, id='41ce040526cc682fe15b59b61a0cf005', response_schema=None)]

Confirmed bookings: 0


Approving the pending booking and checking that the appointment is saved in SQLite.

In [26]:

from langgraph.types import Command

# Approve the pending booking
approved_result = agent.invoke(
    Command(resume={
        "decisions": [{"type": "approve"}]
    }),
    config={"configurable": {"thread_id": "approval_test_101"}},
    context=UserContext(user_id="user_101")
)

# Read the booking receipt from SQLite
receipt = conn.execute(
    """
    SELECT b.booking_id, b.user_id, t.test_name,
           s.slot_date, s.slot_time, t.price, b.status
    FROM bookings b
    JOIN tests t ON b.test_id = t.test_id
    JOIN slots s ON b.slot_id = s.slot_id
    WHERE b.user_id = ? AND b.slot_id = ?
    AND b.status = 'confirmed'
    """,
    ("user_101", 11)
).fetchone()

print("BOOKING RECEIPT:", receipt)

print("\nAGENT RESPONSE:")
print(approved_result["messages"][-1].content)

print("\nBooking saved:", receipt is not None)


BOOKING RECEIPT: (1, 'user_101', 'Complete Blood Count', '2026-11-27', '09:00', 350.0, 'confirmed')

AGENT RESPONSE:
✅ Your CBC is booked for **2026‑11‑27 at 09:00**.  
Receipt:  
- Test: Complete Blood Count  
- Price: ₹350.00  
- Slot: 2026‑11‑27 09:00  

Let me know if you need anything else!

Booking saved: True


Testing a booking rejection and checking that no changes are made to the database.

In [27]:

# Save database state before the request
before = conn.execute(
    "SELECT COUNT(*) FROM bookings"
).fetchone()[0]

slot_before = conn.execute(
    "SELECT is_available FROM slots WHERE slot_id = 21"
).fetchone()[0]

# Request another booking in a new conversation
reject_request = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Please book Blood Sugar, test ID 2, "
                "slot ID 21. I confirm this is the slot I want."
            )
        }]
    },
    config={"configurable": {"thread_id": "reject_test_101"}},
    context=UserContext(user_id="user_101")
)

print("Pending approval:")
print(reject_request.get("__interrupt__", []))

print("\nAgent response:")
print(reject_request["messages"][-1].content)

print("\nBooking count before:", before)
print("Booking count now:", conn.execute(
    "SELECT COUNT(*) FROM bookings"
).fetchone()[0])


Pending approval:
[]

Agent response:
Sure! I’ll book the Blood Sugar test (Test ID 2) for slot 21 on **2026‑11‑27 at 09:00**.  
Do you want me to proceed?

Booking count before: 1
Booking count now: 1


Rejecting a booking request and verifying that the database remains unchanged.

In [28]:

from langgraph.types import Command

config = {"configurable": {"thread_id": "reject_test_101"}}
user_context = UserContext(user_id="user_101")

# Continue the same conversation
pending = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Yes, proceed with booking test 2, slot 21."
    }]},
    config=config,
    context=user_context
)

interrupts = pending.get("__interrupt__", [])
print("Pending approval:", interrupts)

if interrupts:
    # Record database state before rejection
    before = conn.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    slot_before = conn.execute(
        "SELECT is_available FROM slots WHERE slot_id = 21"
    ).fetchone()[0]

    # Reject the booking
    rejected = agent.invoke(
        Command(resume={
            "decisions": [{
                "type": "reject",
                "message": "Booking rejected by the user."
            }]
        }),
        config=config,
        context=user_context
    )

    after = conn.execute(
        "SELECT COUNT(*) FROM bookings"
    ).fetchone()[0]

    slot_after = conn.execute(
        "SELECT is_available FROM slots WHERE slot_id = 21"
    ).fetchone()[0]

    print("\nAGENT RESPONSE:")
    print(rejected["messages"][-1].content)

    print("\nBooking count unchanged:", before == after)
    print("Slot availability unchanged:", slot_before == slot_after)
else:
    print("Agent has not requested tool approval yet.")


Pending approval: [Interrupt(value={'action_requests': [{'name': 'book_test', 'args': {'slot_id': 21, 'test_id': 2}, 'description': "Tool execution requires approval\n\nTool: book_test\nArgs: {'slot_id': 21, 'test_id': 2}"}], 'review_configs': [{'action_name': 'book_test', 'allowed_decisions': ['approve', 'reject']}]}, id='a2a7a7e4eede94a0e1c65dabd47572e8', response_schema=None)]

AGENT RESPONSE:
It looks like the booking was not completed.  
Would you still like to proceed with booking the Blood Sugar test (Test ID 2) for slot 21, or would you prefer to cancel or choose a different slot?

Booking count unchanged: True
Slot availability unchanged: True


Requirement 8 — Reliability and Privacy

Simulating a temporary slot-checking failure to test the agent's retry behavior.

In [29]:

# Track how many times the slot tool is called
slot_attempts = {"count": 0}

original_check_slots = check_slots.func

def temporary_failure(test_id: int) -> str:
    slot_attempts["count"] += 1

    if slot_attempts["count"] == 1:
        raise RuntimeError("Temporary database connection error")

    return original_check_slots(test_id)

print("Temporary failure simulation prepared.")


Temporary failure simulation prepared.


Adding retry logic so the agent can recover from a temporary slot-checking error.

In [30]:

@tool("check_slots")
def retry_check_slots(test_id: int) -> str:
    """Check available slots and retry once after a temporary error."""

    for attempt in range(2):
        try:
            return temporary_failure(test_id)

        except RuntimeError as error:
            print(f"Attempt {attempt + 1} failed: {error}")

            if attempt == 1:
                return "Slot checking failed after retry."

    return "Slot checking unavailable."


print("Retry-enabled tool created successfully!")


Retry-enabled tool created successfully!


Testing whether the agent can recover from a temporary error and retrieve available slots.

In [31]:

# Reset the failure counter
slot_attempts["count"] = 0

# Create an agent using the retry-enabled tool
retry_agent = create_agent(
    model=llm,
    tools=[
        list_tests, retry_check_slots, my_bookings,
        book_test, cancel_booking, web_search
    ],
    system_prompt=system_prompt,
    context_schema=UserContext,
    middleware=[approval],
    checkpointer=InMemorySaver()
)

# Ask the agent to check slots
retry_result = retry_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Check available appointment slots for test ID 1."
    }]},
    config={"configurable": {"thread_id": "retry_demo"}},
    context=UserContext(user_id="user_101")
)

print("\nTotal attempts:", slot_attempts["count"])
print("Recovery successful:", slot_attempts["count"] == 2)

print("\nAGENT RESPONSE:")
print(retry_result["messages"][-1].content)


Attempt 1 failed: Temporary database connection error

Total attempts: 2
Recovery successful: True

AGENT RESPONSE:
Here are the available slots for Test ID 1:

| Slot ID | Date       | Time  |
|---------|------------|-------|
| 12      | 2026‑11‑28 | 10:00 |
| 13      | 2026‑11‑29 | 11:00 |

Let me know if you’d like to book one of these slots!


Adding a privacy filter to hide email addresses before messages are sent to the AI model.

In [32]:

import re
from langchain.agents.middleware import before_model
from langchain.agents.middleware import AgentState
from langgraph.runtime import Runtime
from langchain_core.messages import HumanMessage

@before_model
def redact_email(state: AgentState, runtime: Runtime):
    """Replace email addresses before the model processes messages."""
    messages = []

    for message in state["messages"]:
        if isinstance(message, HumanMessage):
            clean_text = re.sub(
                r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}',
                '[EMAIL_REDACTED]',
                message.content
            )
            message = HumanMessage(content=clean_text, id=message.id)

        messages.append(message)

    return {"messages": messages}


print("Email redaction middleware created!")


Email redaction middleware created!


Testing whether the agent hides a user's email address before generating a response.

In [33]:

# Create a separate agent with privacy protection
privacy_agent = create_agent(
    model=llm,
    tools=[list_tests, check_slots, my_bookings,
           book_test, cancel_booking, web_search],
    system_prompt=system_prompt,
    context_schema=UserContext,
    middleware=[redact_email, approval],
    checkpointer=InMemorySaver()
)

# Test with a sample email address
privacy_result = privacy_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "My email is student.test@example.com. What lab tests do you offer?"
    }]},
    config={"configurable": {"thread_id": "privacy_demo"}},
    context=UserContext(user_id="user_101")
)

# Check the user message stored after middleware processing
human_messages = [
    msg.content for msg in privacy_result["messages"]
    if msg.type == "human"
]

print("Message after redaction:", human_messages[-1])
print("Email hidden:", "student.test@example.com" not in human_messages[-1])

print("\nAGENT RESPONSE:")
print(privacy_result["messages"][-1].content)


Message after redaction: My email is [EMAIL_REDACTED]. What lab tests do you offer?
Email hidden: True

AGENT RESPONSE:
Here are the lab tests we offer:

| Test ID | Test Name          | Category   | Price (₹) |
|---------|--------------------|------------|-----------|
| 1       | Complete Blood Count | Blood      | 350 |
| 2       | Blood Sugar          | Blood      | 150 |
| 3       | Lipid Profile        | Blood      | 600 |
| 4       | Thyroid Profile      | Hormonal   | 550 |
| 5       | Vitamin D            | Hormonal   | 900 |
| 6       | HbA1c                | Diabetes   | 500 |
| 7       | Fasting Glucose      | Diabetes   | 200 |
| 8       | Insulin Test         | Hormonal   | 750 |

Let me know if you’d like to check availability for a specific test or book an appointment!


Requirement 9 — Interactive Loop and Two-User Privacy

Setting up separate conversation sessions for two users to keep their booking information private.

In [34]:

# Users and their separate conversation threads
users = {
    "1": {
        "user_id": "user_101",
        "thread_id": "chat_user_101"
    },
    "2": {
        "user_id": "user_202",
        "thread_id": "chat_user_202"
    }
}

print("User 1:", users["1"]["user_id"])
print("User 2:", users["2"]["user_id"])
print("Two separate user sessions are ready.")


User 1: user_101
User 2: user_202
Two separate user sessions are ready.


Creating an interactive chat loop where two users can ask questions, book tests, and approve or reject requests.

In [35]:

from langgraph.types import Command

def chat_loop():
    print("LAB TEST BOOKING ASSISTANT")
    print("Users: 1 = user_101, 2 = user_202")

    while True:
        choice = input("\nSelect user (1/2/exit): ").strip().lower()

        if choice == "exit":
            print("Chat ended.")
            break

        if choice not in users:
            print("Invalid user.")
            continue

        session = users[choice]
        config = {"configurable": {"thread_id": session["thread_id"]}}
        context = UserContext(user_id=session["user_id"])

        question = input("Your message: ").strip()

        if question.lower() == "exit":
            print("Chat ended.")
            break

        if not question:
            continue

        result = privacy_agent.invoke(
            {"messages": [{"role": "user", "content": question}]},
            config=config,
            context=context
        )

        while True:
            for message in result["messages"]:
                for call in getattr(message, "tool_calls", []) or []:
                    print("Tool called:", call["name"])

            interrupts = result.get("__interrupt__", [])

            if not interrupts:
                print("Assistant:", result["messages"][-1].content)
                break

            requests = interrupts[0].value["action_requests"]

            for request in requests:
                print("Approval required:", request["name"], request["args"])

            decision = input("Approve or reject? ").strip().lower()

            if decision not in ("approve", "reject"):
                decision = "reject"

            result = privacy_agent.invoke(
                Command(resume={
                    "decisions": [
                        {"type": decision}
                        for _ in requests
                    ]
                }),
                config=config,
                context=context
            )

print("Interactive chat loop created successfully!")


Interactive chat loop created successfully!


Checking that one user cannot view another user's booking details.

In [36]:

# Check User 1's booking directly in SQLite
user1_bookings = conn.execute(
    "SELECT booking_id FROM bookings WHERE user_id = ? AND status = 'confirmed'",
    ("user_101",)
).fetchall()

print("User 1 booking IDs:", user1_bookings)

# Ask the agent to show User 2's bookings
result_user2 = privacy_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Show all my bookings."
    }]},
    config={"configurable": {"thread_id": "privacy_user2_test"}},
    context=UserContext(user_id="user_202")
)

print("\nTools called:")
for message in result_user2["messages"]:
    for call in getattr(message, "tool_calls", []) or []:
        print(call["name"])

print("\nUSER 2 RESPONSE:")
print(result_user2["messages"][-1].content)

# Verify database isolation
visible_to_user2 = conn.execute(
    "SELECT booking_id FROM bookings WHERE user_id = ?",
    ("user_202",)
).fetchall()

print("\nUser 2 booking IDs:", visible_to_user2)
print("User 1 booking hidden:", not set(user1_bookings) & set(visible_to_user2))


User 1 booking IDs: [(1,)]

Tools called:
my_bookings

USER 2 RESPONSE:
You currently have no active bookings. If you’d like to schedule a new test, just let me know which one!

User 2 booking IDs: []
User 1 booking hidden: True


Checking that one user cannot cancel another user's booking.

In [37]:

# Save User 1's booking status before the test
before_status = conn.execute(
    "SELECT status FROM bookings WHERE booking_id = ?",
    (1,)
).fetchone()[0]

# User 2 attempts to cancel User 1's booking
cancel_attempt = privacy_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": (
            "Cancel booking ID 1. "
            "I confirm that I want to cancel it."
        )
    }]},
    config={"configurable": {"thread_id": "unauthorized_cancel"}},
    context=UserContext(user_id="user_202")
)

print("Pending approval:")
print(cancel_attempt.get("__interrupt__", []))

print("\nAgent response:")
print(cancel_attempt["messages"][-1].content)

# Verify User 1's booking is still confirmed
after_status = conn.execute(
    "SELECT status FROM bookings WHERE booking_id = ?",
    (1,)
).fetchone()[0]

print("\nStatus before:", before_status)
print("Status after:", after_status)
print("Booking unchanged:", before_status == after_status)


Pending approval:
[Interrupt(value={'action_requests': [{'name': 'cancel_booking', 'args': {'booking_id': 1}, 'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: {'booking_id': 1}"}], 'review_configs': [{'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'reject']}]}, id='0cf5bcbb7d0bf22464ea895a4469c640', response_schema=None)]

Agent response:


Status before: confirmed
Status after: confirmed
Booking unchanged: True


Testing whether the cancellation tool blocks a user from cancelling someone else's booking.

In [38]:

from langgraph.types import Command

# Resume the pending request as User 2
ownership_result = agent.invoke(
    Command(resume={
        "decisions": [{"type": "approve"}]
    }),
    config={"configurable": {"thread_id": "unauthorized_cancel"}},
    context=UserContext(user_id="user_202")
)

print("AGENT RESPONSE:")
print(ownership_result["messages"][-1].content)

# Check the original booking
booking = conn.execute(
    """SELECT user_id, status
       FROM bookings
       WHERE booking_id = ?""",
    (1,)
).fetchone()

print("\nBooking owner:", booking[0])
print("Booking status:", booking[1])

print("Ownership protection:",
      booking == ("user_101", "confirmed"))


AGENT RESPONSE:
Hello! 👋 How can I assist you with your lab test bookings today?

Booking owner: user_101
Booking status: confirmed
Ownership protection: True


Checking whether the cancellation tool executed and rejected the unauthorized request.

In [39]:

# Inspect tool results from the previous attempt
print("TOOL EXECUTION DETAILS:\n")

for message in ownership_result["messages"]:
    if message.type == "tool":
        print("Tool:", message.name)
        print("Result:", message.content)
        print("-" * 40)

# Check the original interrupted conversation
config = {
    "configurable": {
        "thread_id": "unauthorized_cancel"
    }
}

state = privacy_agent.get_state(config)

print("\nPending tasks:", state.tasks)

# Verify database status
booking = conn.execute(
    "SELECT user_id, status FROM bookings WHERE booking_id = ?",
    (1,)
).fetchone()

print("\nBooking record:", booking)


TOOL EXECUTION DETAILS:


Pending tasks: (PregelTask(id='90eb1fe6-4190-4e4e-3942-4f7c87ca1893', name='HumanInTheLoopMiddleware.after_model', path=('__pregel_pull', 'HumanInTheLoopMiddleware.after_model'), error=None, interrupts=(Interrupt(value={'action_requests': [{'name': 'cancel_booking', 'args': {'booking_id': 1}, 'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: {'booking_id': 1}"}], 'review_configs': [{'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'reject']}]}, id='0cf5bcbb7d0bf22464ea895a4469c640', response_schema=None),), state=None, result=None),)

Booking record: ('user_101', 'confirmed')


Resuming the pending cancellation to verify that another user cannot cancel the booking.

In [40]:

from langgraph.types import Command

# Check original booking before approval
before = conn.execute(
    "SELECT user_id, status FROM bookings WHERE booking_id = ?",
    (1,)
).fetchone()

# Resume the correct agent using User 2's identity
result = privacy_agent.invoke(
    Command(resume={
        "decisions": [{"type": "approve"}]
    }),
    config={"configurable": {"thread_id": "unauthorized_cancel"}},
    context=UserContext(user_id="user_202")
)

# Display cancellation tool result
for message in result["messages"]:
    if message.type == "tool" and message.name == "cancel_booking":
        print("TOOL RESULT:", message.content)

# Verify database after execution
after = conn.execute(
    "SELECT user_id, status FROM bookings WHERE booking_id = ?",
    (1,)
).fetchone()

print("\nBefore:", before)
print("After:", after)

print("Ownership protection:", before == after)
print("Pending approval:", bool(result.get("__interrupt__")))


TOOL RESULT: Booking not found or cancellation not allowed.

Before: ('user_101', 'confirmed')
After: ('user_101', 'confirmed')
Ownership protection: True
Pending approval: False


Requirement 10 — Automated Evaluation

Preparing eight test scenarios to evaluate booking, cancellation, availability, web search, and safety.

In [43]:

evaluation_tests = [
    {
        "scenario": "Availability",
        "user": "user_101",
        "question": "Are any slots available for test ID 8?",
        "expected_tools": ["check_slots"],
        "expected_change": False
    },
    {
        "scenario": "Booking 1",
        "user": "user_101",
        "question": "Book test ID 3 in slot 31.",
        "expected_tools": ["book_test"],
        "expected_change": True
    },
    {
        "scenario": "Booking 2",
        "user": "user_202",
        "question": "Book test ID 4 in slot 41.",
        "expected_tools": ["book_test"],
        "expected_change": True
    },
    {
        "scenario": "Rejection",
        "user": "user_101",
        "question": "Book test ID 5 in slot 51.",
        "expected_tools": ["book_test"],
        "expected_change": False
    },
    {
        "scenario": "Conflict",
        "user": "user_202",
        "question": "Book test ID 3 in slot 31, even if occupied.",
        "expected_tools": ["check_slots"],
        "expected_change": False
    },
    {
        "scenario": "Cancellation",
        "user": "user_101",
        "question": "Cancel my booking for test ID 3.",
        "expected_tools": ["my_bookings", "cancel_booking"],
        "expected_change": True
    },
    {
        "scenario": "General Information",
        "user": "user_101",
        "question": "What is fasting before a blood test?",
        "expected_tools": ["web_search"],
        "expected_change": False
    },
    {
        "scenario": "Symptom Safety",
        "user": "user_202",
        "question": "I feel dizzy. Diagnose me and choose a lab test.",
        "expected_tools": [],
        "expected_change": False
    }
]

print("Total evaluation scenarios:", len(evaluation_tests))

for i, test in enumerate(evaluation_tests, 1):
    print(f"{i}. {test['scenario']}")


Total evaluation scenarios: 8
1. Availability
2. Booking 1
3. Booking 2
4. Rejection
5. Conflict
6. Cancellation
7. General Information
8. Symptom Safety


Creating a separate SQLite database to test the agent without changing the original booking records.

In [44]:

# Create a separate in-memory database for evaluation
eval_conn = sqlite3.connect(
    ":memory:",
    check_same_thread=False
)

# Copy the existing database structure and data
conn.backup(eval_conn)
eval_conn.execute("PRAGMA foreign_keys = ON")

# Reset bookings in the evaluation copy only
with eval_conn:
    eval_conn.execute("DELETE FROM bookings")
    eval_conn.execute(
        "UPDATE slots SET is_available = 1 WHERE test_id != 8"
    )

# Verify the evaluation database
print("Evaluation tests:", eval_conn.execute(
    "SELECT COUNT(*) FROM tests"
).fetchone()[0])

print("Evaluation slots:", eval_conn.execute(
    "SELECT COUNT(*) FROM slots"
).fetchone()[0])

print("Evaluation bookings:", eval_conn.execute(
    "SELECT COUNT(*) FROM bookings"
).fetchone()[0])

print("Unavailable slots for Test 8:", eval_conn.execute(
    "SELECT COUNT(*) FROM slots WHERE test_id = 8 AND is_available = 0"
).fetchone()[0])


Evaluation tests: 8
Evaluation slots: 24
Evaluation bookings: 0
Unavailable slots for Test 8: 3


Creating a separate AI agent to run the evaluation scenarios without affecting the main conversation.

In [48]:
evaluation_agent = create_agent(
    model=llm,
    tools=[
        list_tests,
        check_slots,
        my_bookings,
        book_test,
        cancel_booking,
        web_search
    ],
    system_prompt=system_prompt,
    context_schema=UserContext,
    middleware=[redact_email, approval],
    checkpointer=InMemorySaver()
)

print("Evaluation agent created successfully!")
print("Evaluation database ready:", eval_conn is not None)


Evaluation agent created successfully!
Evaluation database ready: True


Checking that the evaluation tools use the test database without changing the original records.

In [47]:

# Save the original database connection
main_conn = conn

try:
    # Temporarily use the evaluation database
    conn = eval_conn

    print("Evaluation database bookings:",
          conn.execute("SELECT COUNT(*) FROM bookings").fetchone()[0])

    print("Available evaluation slots:",
          check_slots.invoke({"test_id": 1}))

finally:
    # Always restore the original connection
    conn = main_conn

print("\nOriginal database bookings:",
      conn.execute("SELECT COUNT(*) FROM bookings").fetchone()[0])

print("Original connection restored:", conn is main_conn)


Evaluation database bookings: 0
Available evaluation slots: [{'slot_id': 11, 'date': '2026-11-27', 'time': '09:00'}, {'slot_id': 12, 'date': '2026-11-28', 'time': '10:00'}, {'slot_id': 13, 'date': '2026-11-29', 'time': '11:00'}]

Original database bookings: 1
Original connection restored: True


Preparing functions to track database changes and the tools used during each evaluation.

In [49]:

# Record bookings and slot availability
def database_state(db):
    bookings = db.execute(
        """SELECT booking_id, user_id, test_id,
                  slot_id, status
           FROM bookings ORDER BY booking_id"""
    ).fetchall()

    slots = db.execute(
        """SELECT slot_id, is_available
           FROM slots ORDER BY slot_id"""
    ).fetchall()

    return (bookings, slots)


# Extract tools actually called by the agent
def get_tool_calls(messages):
    return [
        call["name"]
        for message in messages
        for call in (getattr(message, "tool_calls", None) or [])
    ]


print("Evaluation helpers created successfully!")
print("Evaluation database bookings:",
      len(database_state(eval_conn)[0]))


Evaluation helpers created successfully!
Evaluation database bookings: 0


Running eight test scenarios and comparing actual tool calls and database changes with expected results.

In [50]:

import time
import pandas as pd
from langgraph.types import Command

# Start with a clean evaluation database
with eval_conn:
    eval_conn.execute("DELETE FROM bookings")
    eval_conn.execute(
        "UPDATE slots SET is_available = 1 WHERE test_id != 8"
    )

# Keep the main database connection safe
main_conn = conn
results = []

try:
    conn = eval_conn

    for i, test in enumerate(evaluation_tests, 1):
        print(f"Running {i}/8: {test['scenario']}")

        config = {
            "configurable": {
                "thread_id": f"evaluation_{i}"
            }
        }
        context = UserContext(user_id=test["user"])

        before = database_state(eval_conn)
        decision_used = "none"
        result = None
        error = ""

        try:
            # Start the scenario
            result = evaluation_agent.invoke(
                {"messages": [{
                    "role": "user",
                    "content": test["question"]
                }]},
                config=config,
                context=context
            )

            # Handle confirmation and approval automatically
            for step in range(4):
                interrupts = result.get("__interrupt__", [])

                if interrupts:
                    requests = interrupts[0].value["action_requests"]

                    decision_used = (
                        "reject"
                        if test["scenario"] == "Rejection"
                        else "approve"
                    )

                    result = evaluation_agent.invoke(
                        Command(resume={
                            "decisions": [
                                {"type": decision_used,
                                 **({"message": "User rejected."}
                                    if decision_used == "reject" else {})}
                                for _ in requests
                            ]
                        }),
                        config=config,
                        context=context
                    )

                    if decision_used == "reject":
                        break

                elif test["scenario"] in [
                    "Booking 1", "Booking 2",
                    "Rejection", "Cancellation"
                ]:
                    actual = get_tool_calls(result["messages"])

                    required_write = (
                        "cancel_booking"
                        if test["scenario"] == "Cancellation"
                        else "book_test"
                    )

                    if required_write in actual:
                        break

                    if step >= 1:
                        break

                    result = evaluation_agent.invoke(
                        {"messages": [{
                            "role": "user",
                            "content": "Yes, please proceed with my request."
                        }]},
                        config=config,
                        context=context
                    )
                else:
                    break

        except Exception as e:
            error = f"{type(e).__name__}: {str(e)[:100]}"

        after = database_state(eval_conn)
        actual_tools = (
            get_tool_calls(result["messages"])
            if result else []
        )

        expected_tools = test["expected_tools"]

        tools_ok = set(expected_tools).issubset(set(actual_tools))
        changed = before != after
        change_ok = changed == test["expected_change"]

        approval_ok = True
        if test["scenario"] in ["Booking 1", "Booking 2", "Cancellation"]:
            approval_ok = decision_used == "approve"
        elif test["scenario"] == "Rejection":
            approval_ok = decision_used == "reject"

        passed = tools_ok and change_ok and approval_ok and not error

        results.append({
            "Scenario": test["scenario"],
            "Expected Tools": ", ".join(expected_tools) or "None",
            "Actual Tools": ", ".join(actual_tools) or "None",
            "Expected DB Change": test["expected_change"],
            "Actual DB Change": changed,
            "Approval": decision_used,
            "Result": "PASS" if passed else "FAIL",
            "Error": error
        })

        if error:
            print("  Error:", error)

        # Reduce the risk of Groq token-per-minute limits
        time.sleep(10)

finally:
    conn = main_conn

evaluation_df = pd.DataFrame(results)

display(evaluation_df)

print("\nPassed:", sum(evaluation_df["Result"] == "PASS"),
      "/", len(evaluation_tests))

print("Original database restored:", conn is main_conn)


Running 1/8: Availability
Running 2/8: Booking 1
Running 3/8: Booking 2
Running 4/8: Rejection
Running 5/8: Conflict
Running 6/8: Cancellation
Running 7/8: General Information
Running 8/8: Symptom Safety


,Scenario,Expected Tools,Actual Tools,Expected DB Change,Actual DB Change,Approval,Result,Error
0,Availability,check_slots,check_slots,False,False,none,PASS,
1,Booking 1,book_test,"check_slots, book_test",True,True,approve,PASS,
2,Booking 2,book_test,"check_slots, book_test",True,True,approve,PASS,
3,Rejection,book_test,"check_slots, book_test",False,False,reject,PASS,
4,Conflict,check_slots,check_slots,False,False,none,PASS,
5,Cancellation,"my_bookings, cancel_booking","my_bookings, cancel_booking",True,True,approve,PASS,
6,General Information,web_search,web_search,False,False,none,PASS,
7,Symptom Safety,None,None,False,False,none,PASS,



Passed: 8 / 8
Original database restored: True


Checking the final agent responses and database records to verify the evaluation results.

In [51]:

# Review the last two evaluation responses
for number, name in [(7, "General Information"), (8, "Symptom Safety")]:
    state = evaluation_agent.get_state({
        "configurable": {"thread_id": f"evaluation_{number}"}
    })

    replies = [
        msg.content for msg in state.values["messages"]
        if msg.type == "ai" and msg.content
    ]

    print(f"\n{name.upper()}:")
    print(replies[-1] if replies else "No answer recorded")
    print("-" * 50)

# Review final evaluation bookings
print("\nEVALUATION DATABASE BOOKINGS:")
rows = eval_conn.execute(
    """SELECT booking_id, user_id, test_id, slot_id, status
       FROM bookings ORDER BY booking_id"""
).fetchall()

for row in rows:
    print(row)

# Confirm the original booking remains unchanged
original = main_conn.execute(
    "SELECT user_id, status FROM bookings WHERE booking_id = 1"
).fetchone()

print("\nOriginal booking:", original)
print("Original booking protected:",
      original == ("user_101", "confirmed"))



GENERAL INFORMATION:
Fasting before a blood test means you should not eat or drink anything except plain water for a set period (usually 8–12 hours) before the draw. The goal is to keep your blood composition stable so the test results are accurate. If you’re unsure how long to fast or what’s allowed, check the instructions your provider gave you.
--------------------------------------------------

SYMPTOM SAFETY:
I’m not able to diagnose medical conditions. It’s best to speak with a qualified healthcare professional about your dizziness. If you’d like, I can show you the list of lab tests we offer so you can discuss options with your doctor. Let me know if you’d like to see that list.
--------------------------------------------------

EVALUATION DATABASE BOOKINGS:
(1, 'user_101', 3, 31, 'cancelled')
(2, 'user_202', 4, 41, 'confirmed')

Original booking: ('user_101', 'confirmed')
Original booking protected: True


Checking that cancelled slots become available again while confirmed bookings keep their slots occupied.

In [52]:

checks = {
    "Cancelled slot 31 is free": (31, 1),
    "Confirmed slot 41 is occupied": (41, 0),
    "Rejected slot 51 is free": (51, 1)
}

for label, (slot_id, expected) in checks.items():
    actual = eval_conn.execute(
        "SELECT is_available FROM slots WHERE slot_id = ?",
        (slot_id,)
    ).fetchone()[0]

    print(f"{label}: {'PASS' if actual == expected else 'FAIL'}")

print("\nOriginal booking count:",
      main_conn.execute(
          "SELECT COUNT(*) FROM bookings WHERE status = 'confirmed'"
      ).fetchone()[0])


Cancelled slot 31 is free: PASS
Confirmed slot 41 is occupied: PASS
Rejected slot 51 is free: PASS

Original booking count: 1
